In [1]:
import sqlite3
import pandas as pd

In [2]:
# 1. Connect to an in-memory SQLite database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [3]:
# 2. Create the tables
cursor.execute("""
CREATE TABLE experiments (
    experiment_id INTEGER PRIMARY KEY,
    title TEXT,
    duration_minutes INTEGER
);
""")

cursor.execute("""
CREATE TABLE materials (
    material_id INTEGER PRIMARY KEY,
    experiment_id INTEGER,
    material_name TEXT,
    FOREIGN KEY (experiment_id) REFERENCES experiments (experiment_id)
);
""")

In [4]:
# 3. Insert mock data for experiments and materials
experiments_data = [
    (1, "Volcano Eruption", 15),
    (2, "Invisible Ink", 30),
    (3, "Crystal Growing", 120),
    (4, "Magnetic Slime", 20)
]

materials_data = [
    (101, 1, "Baking Soda"),
    (102, 1, "Vinegar"),
    (103, 2, "Lemon Juice"),
    (104, 3, "Borax"),
    (105, 4, "Glue"),
    (106, 4, "Iron Filings")
]

cursor.executemany("INSERT INTO experiments VALUES (?, ?, ?);", experiments_data)
cursor.executemany("INSERT INTO materials VALUES (?, ?, ?);", materials_data)
conn.commit()

print("--- Database successfully initialized with sample data ---\n")


# =========================================================================
# GOAL 1 & 2: Use AS to rename columns and use short aliases for tables in a JOIN
# =========================================================================
print("### Task 1: Joining tables with column and table aliases (AS)")
join_query = """
SELECT 
    e.title AS [Experiment Name], 
    m.material_name AS [Required Material]
FROM experiments AS e
JOIN materials AS m ON e.experiment_id = m.experiment_id;
"""
df_joined = pd.read_sql_query(join_query, conn)
print(df_joined, "\n")


# =========================================================================
# GOAL 3 & 4: Subquery with IN to find matching experiments
# (e.g., Find experiments that use either 'Vinegar' or 'Glue')
# =========================================================================
print("### Task 2: Subquery using IN to find matching experiments")
in_subquery = """
SELECT title AS [Experiment Name], duration_minutes AS [Duration (min)]
FROM experiments
WHERE experiment_id IN (
    SELECT experiment_id 
    FROM materials 
    WHERE material_name = 'Vinegar' OR material_name = 'Glue'
);
"""
df_in = pd.read_sql_query(in_subquery, conn)
print(df_in, "\n")


# =========================================================================
# GOAL 4: Subquery with = to find the quickest experiment
# =========================================================================
print("### Task 3: Subquery using = to find the quickest experiment")
equal_subquery = """
SELECT title AS [Quickest Experiment], duration_minutes AS [Minutes]
FROM experiments
WHERE duration_minutes = (
    SELECT MIN(duration_minutes) 
    FROM experiments
);
"""
df_equal = pd.read_sql_query(equal_subquery, conn)
print(df_equal, "\n")


# Close the connection
conn.close()

--- Database successfully initialized with sample data ---

### Task 1: Joining tables with column and table aliases (AS)
    Experiment Name Required Material
0  Volcano Eruption       Baking Soda
1  Volcano Eruption           Vinegar
2     Invisible Ink       Lemon Juice
3   Crystal Growing             Borax
4    Magnetic Slime              Glue
5    Magnetic Slime      Iron Filings 

### Task 2: Subquery using IN to find matching experiments
    Experiment Name  Duration (min)
0  Volcano Eruption              15
1    Magnetic Slime              20 

### Task 3: Subquery using = to find the quickest experiment
  Quickest Experiment  Minutes
0    Volcano Eruption       15 

